# YH2 M080924J — ROI transmission, normalised to the open beam

Replaces the earlier version, which divided each ROI by the **TZM** ROI. That produced a
ratio of two attenuated beams, not a transmission, and it legitimately exceeded 1 wherever
an ROI attenuated less than the TZM reference path.

This notebook computes **both** quantities and keeps them under distinct names:

| column | meaning | use it for |
|---|---|---|
| `{roi}_T` | transmission against the open beam | absolute work, nbragg fitting, thickness |
| `{roi}_ratio_tzm` | ROI / TZM in the same frame | time series — immune to beam drift |

The open beam is a static `sum_smoothed` file, so `_T` is vulnerable to flux drift across the
27-hour hold. `_ratio_tzm` is not, because TZM cannot change composition and sits in the same
exposure. Cell 10 measures that drift and tells you which to trust.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tifffile import imread
import nbragg
import NCrystal as NC

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

def no_nan_divide(x, y):
    return np.where(y != 0, np.divide(x, np.where(y != 0, y, 1)), 0)

def plot_slice_region(ax, slice_tuple, color="red", label=None):
    """Plot rectangular ROI on image."""
    y0, y1 = slice_tuple[0].start, slice_tuple[0].stop
    x0, x1 = slice_tuple[1].start, slice_tuple[1].stop
    ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0],
            color=color, linewidth=2, label=label)

print("libraries imported")

In [ ]:
df_runs = pd.read_csv('tiff_radiography_holds_final.csv')
df_rois = pd.read_csv('temperature_roi_map.csv')

tiff_dir = Path('M080924J/tiffs')

N_CH = 800
BAND = slice(100, 300)          # the working wavelength band, in channels

# NOTE: 0.004420134940585 A/channel assumes L = 8.95 m, which is the ZrH2 run.
# The flight path for THIS campaign is not established -- the axis is consistent
# between panels but the absolute wavelength is provisional.
wl = (np.arange(N_CH) + 1) * 0.004420134940585

ROI_ALL = ["center_sample", "upper_sample", "sample_edge", "gap", "tzm"]

print(f"Total runs: {len(df_runs)}")
print(df_runs.groupby('temp_bin').size())
print()
print(df_rois[['temp_bin', 'n_runs', 'center_sample_roi', 'sample_roi_center',
               'gap_roi_center', 'tzm_roi_center']])

In [ ]:
def get_roi_slices(temp_bin, df_rois):
    """ROI slices for a temperature bin -- all 5 ROIs."""
    row = df_rois[df_rois['temp_bin'] == temp_bin].iloc[0]
    y_start, y_stop = 162, 378
    roi_width = 24

    return {
        'center_sample': np.s_[205:255,
                               int(row['center_sample_roi'])-25:int(row['center_sample_roi'])+25],
        'upper_sample': np.s_[int(row['upper_sample_y'])-25:int(row['upper_sample_y'])+25,
                              int(row['upper_sample_x'])-25:int(row['upper_sample_x'])+25],
        'sample_edge': np.s_[y_start:y_stop,
                             int(row['sample_roi_center'])-roi_width//2:int(row['sample_roi_center'])+roi_width//2],
        'gap': np.s_[y_start:y_stop,
                     int(row['gap_roi_center'])-roi_width//2:int(row['gap_roi_center'])+roi_width//2],
        'tzm': np.s_[y_start:y_stop,
                     int(row['tzm_roi_center'])-roi_width//2:int(row['tzm_roi_center'])+roi_width//2],
    }

test_rois = get_roi_slices(25, df_rois)
print("ROI slices for temp_bin=25 (RT before):")
for name, roi in test_rois.items():
    print(f"  {name}: {roi}")

In [ ]:
example_runs = {25: 110006, 950: 110214, 951: 111054, 500: 111397, 26: 111569}
colors = {'center_sample': 'yellow', 'upper_sample': 'orange',
          'sample_edge': 'red', 'gap': 'cyan', 'tzm': 'lime'}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (temp_bin, run_num) in enumerate(example_runs.items()):
    ax = axes[i]
    vol = imread(tiff_dir / f"{run_num}_t1e-05_T2400_p1e-07_P100.tiff")
    img = vol.sum(axis=0)
    p5, p95 = np.percentile(img, [5, 95])
    ax.imshow(np.clip((img - p5) / (p95 - p5), 0, 1), cmap='gray', vmin=0, vmax=1)
    for name, roi in get_roi_slices(temp_bin, df_rois).items():
        plot_slice_region(ax, roi, color=colors[name])
    temp_label = '600C' if temp_bin == 500 else df_rois[df_rois['temp_bin']==temp_bin]['temp_range'].iloc[0]
    ax.set_title(f"Run {run_num}\n{temp_label}")
    ax.set_xlabel('x (pixels)')
    if i == 0:
        ax.set_ylabel('y (pixels)')

from matplotlib.patches import Patch
axes[-1].legend(handles=[Patch(facecolor=colors[k], label=k.replace('_',' ').title())
                         for k in colors],
                loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig('roi_overlay_verification.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def extract_roi_spectrum(vol, roi_slice):
    """Spatially averaged spectrum from a 3D volume [wavelength, y, x]."""
    return vol[0:N_CH, roi_slice[0], roi_slice[1]].mean(axis=(1, 2))

spectra_data = []
print(f"extracting {len(ROI_ALL)} ROI spectra from {len(df_runs)} TIFFs...")
for idx, row in df_runs.iterrows():
    run_num, temp_bin = row['run_number'], row['temp_bin']
    tiff_path = tiff_dir / f"{run_num}_t1e-05_T2400_p1e-07_P100.tiff"
    if not tiff_path.exists():
        print(f"  MISSING {run_num}")
        continue
    vol = imread(tiff_path)
    rois = get_roi_slices(temp_bin, df_rois)
    rec = {'run_number': run_num, 'temp_bin': temp_bin,
           'temperature': row['temperature'], 'description': row['description']}
    for roi_name, roi_slice in rois.items():
        rec[roi_name] = extract_roi_spectrum(vol, roi_slice)
    spectra_data.append(rec)
    if (idx + 1) % 10 == 0:
        print(f"  {idx + 1}/{len(df_runs)}")

df_spectra = pd.DataFrame(spectra_data)
print(f"\nextracted {len(df_spectra)} runs x {len(ROI_ALL)} ROIs")
print(f"spectrum length: {df_spectra['tzm'].iloc[0].shape}")

## The fix — normalise to the open beam

The beam profile varies across the frame, and the whole assembly **translates ~140 px between
room temperature and 950 C**, so each ROI must be divided by the open beam sampled *at its own
position, for its own temperature bin*.

In [ ]:
OB_PATH = tiff_dir / "openbeam_furnace_sum_smoothed_t1e-05_T2400_p1e-07_P100.tif"
print("loading open beam:", OB_PATH.name)
vol_ob = imread(OB_PATH)
print("  shape", vol_ob.shape, " dtype", vol_ob.dtype)

ob_spectra = {}
for temp_bin in df_rois["temp_bin"].unique():
    rois = get_roi_slices(temp_bin, df_rois)
    ob_spectra[temp_bin] = {k: extract_roi_spectrum(vol_ob, sl) for k, sl in rois.items()}

print("open-beam ROI spectra for temp bins:", sorted(ob_spectra))
print()
print("open-beam counts per ROI, band median  (a low value means the ROI sits in the penumbra)")
for tb in sorted(ob_spectra):
    cells = "".join("%12.1f" % np.median(ob_spectra[tb][r][BAND]) for r in ROI_ALL)
    print("  temp_bin %-5s %s" % (tb, cells))
print("           " + "".join("%12s" % r[:11] for r in ROI_ALL))

In [ ]:
# ---- transmission against the open beam -> absolute work -----------------
for roi_name in ROI_ALL:
    df_spectra[f"{roi_name}_T"] = df_spectra.apply(
        lambda r: np.divide(
            np.asarray(r[roi_name], dtype=float),
            np.asarray(ob_spectra[r["temp_bin"]][roi_name], dtype=float),
            out=np.full(N_CH, np.nan),
            where=np.asarray(ob_spectra[r["temp_bin"]][roi_name]) > 0), axis=1)

    df_spectra[f"{roi_name}_T_err"] = df_spectra.apply(
        lambda r: np.abs(np.asarray(r[f"{roi_name}_T"], dtype=float)) * np.sqrt(
            1.0 / np.maximum(np.asarray(r[roi_name], dtype=float), 1.0) +
            1.0 / np.maximum(np.asarray(ob_spectra[r["temp_bin"]][roi_name], dtype=float), 1.0)
        ), axis=1)

# ---- ratio to TZM in the same frame -> drift-immune time series ----------
for roi_name in ROI_ALL:
    df_spectra[f"{roi_name}_ratio_tzm"] = df_spectra.apply(
        lambda r: np.divide(np.asarray(r[roi_name], dtype=float),
                            np.asarray(r["tzm"], dtype=float),
                            out=np.full(N_CH, np.nan),
                            where=np.asarray(r["tzm"]) > 0), axis=1)

print("built  _T  (vs open beam)  and  _ratio_tzm  (vs TZM, same frame)")

In [ ]:
# Drop the old TZM-normalised columns that were MISNAMED as transmission.
# Anything downstream still using them will now raise KeyError -- which is the
# point: a stale reference would silently return the wrong quantity.
stale = [c for c in df_spectra.columns if c.endswith(("_transmission", "_trans_err"))]
if stale:
    print("dropping", len(stale), "stale columns:")
    for c in stale:
        print("  ", c)
    df_spectra = df_spectra.drop(columns=stale)
else:
    print("no stale columns present")

## Diagnostics

The open beam is a **sum over many runs**, so its counts far exceed any single sample run and
`_T` lands at the exposure ratio rather than at 1. Two things to check:

1. no ROI exceeds the least-attenuated one
2. the least-attenuated ROI is **flat in wavelength** — if it slopes, the sample and open-beam
   spectra differ and a scalar normalisation will not fix it

In [ ]:
PANELS = [
    ("RT Before Heating",       "temp_bin",   25),
    ("950 C Hold - First Run",  "run_number", 110214),
    ("950 C Hold - Last Run",   "run_number", 111054),
    ("RT After Heating",        "temp_bin",   26),
]

def get_row(col, val):
    sub = df_spectra[df_spectra[col] == val]
    return None if len(sub) == 0 else sub.iloc[0]

print("median sample/open in the band")
print("panel                  run     " + "".join("%12s" % r[:11] for r in ROI_ALL))
for label, col, val in PANELS:
    row = get_row(col, val)
    if row is None:
        print("  %-20s MISSING" % label); continue
    cells = "".join("%12.4f" % np.nanmedian(np.asarray(row[f"{r}_T"], dtype=float)[BAND])
                    for r in ROI_ALL)
    print("  %-20s %-7s %s" % (label, row["run_number"], cells))

row = get_row(*PANELS[0][1:])
g = np.asarray(row["gap_T"], dtype=float)
sel = np.isfinite(g) & (np.arange(N_CH) > 60) & (np.arange(N_CH) < 700)
slope = np.polyfit(wl[sel], g[sel], 1)[0]
med = float(np.nanmedian(g[sel]))
print()
print("gap ROI, 0.27-3.1 A:  median %.4f   slope %+.2f %% end-to-end"
      % (med, 100 * slope * (wl[700] - wl[60]) / med))
print("  flat  -> pure exposure ratio, safe to scale by it")
print("  slope -> spectral mismatch between sample and open-beam runs")

# is there cleaner open beam elsewhere in the frame?
smp_img = imread(tiff_dir / "110006_t1e-05_T2400_p1e-07_P100.tiff")[BAND].sum(axis=0).astype(float)
ob_img = vol_ob[BAND].sum(axis=0).astype(float)
rimg = np.divide(smp_img, ob_img, out=np.full(smp_img.shape, np.nan), where=ob_img > 0)
print()
print("frame-wide 99th pct of sample/open : %.4f" % np.nanpercentile(rimg, 99))
print("gap ROI median                     : %.4f" % med)
print("  if the frame value is much higher, the gap ROI is not truly open")

In [ ]:
# TZM cannot change composition. Whatever it does over 27 hours is instrumental,
# and it sets the floor on any sample change you can claim.
hold = df_spectra[df_spectra["temp_bin"].isin([950, 951])].sort_values("run_number")
tz = np.array([np.nanmedian(np.asarray(r["tzm_T"], dtype=float)[BAND])
               for _, r in hold.iterrows()])

print("TZM transmission across the 950 C hold,", len(tz), "runs")
print("  first %.4f   last %.4f" % (tz[0], tz[-1]))
print("  drift %+.2f %%   rms %.2f %%"
      % (100 * (tz[-1] / tz[0] - 1), 100 * np.std(tz) / np.mean(tz)))
print()
print("  under ~1 %  -> open-beam normalisation is safe, use _T throughout")
print("  several %   -> use _ratio_tzm for the time series; _T carries beam drift")

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(hold["run_number"].values, tz, "o-", ms=4, color="#5E6E3A")
ax.axhline(np.mean(tz), color="0.5", ls="--", lw=1.0)
ax.set_xlabel("run number")
ax.set_ylabel("TZM transmission")
ax.set_title("drift control -- TZM composition is constant by construction",
             fontsize=10, loc="left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Spectra overview

Four stages. With open-beam normalisation every ROI should sit at or below the exposure
ratio — nothing above it, and nothing above 1 once scaled.

In [ ]:
roi_plot = ["center_sample", "upper_sample", "sample_edge", "gap", "tzm"]
colors_plot = {"center_sample": "blue", "upper_sample": "orange",
               "sample_edge": "red", "gap": "green", "tzm": "purple"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
report = []

for ax, (label, col, val) in zip(axes, PANELS):
    row = get_row(col, val)
    if row is None:
        ax.text(0.5, 0.5, "no run matching\n%s == %s" % (col, val),
                ha="center", va="center", transform=ax.transAxes, fontsize=11)
        ax.set_axis_off()
        report.append((label, None, None))
        continue

    worst = 0.0
    for roi in roi_plot:
        t = np.asarray(row["%s_T" % roi], dtype=float)
        worst = max(worst, float(np.nanmax(t[BAND])))
        lw = 2.4 if roi in ("gap", "tzm") else 1.1
        ax.plot(wl[:len(t)], t, label=roi.replace("_", " ").title(),
                color=colors_plot[roi], linewidth=lw)

    ref = float(np.nanmedian(np.asarray(row["gap_T"], dtype=float)[BAND]))
    ax.axhline(ref, color="0.4", ls="--", lw=1.0)
    ax.set_xlabel("Wavelength (A)")
    ax.set_ylabel("Transmission vs open beam")
    title = "%s (Run %s)" % (label, row["run_number"])
    if worst > ref * 1.02:
        title += "\nmax %.4f exceeds the gap ROI %.4f" % (worst, ref)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, max(1.05 * ref, 1.05 * worst))
    ax.grid(alpha=0.3)
    report.append((label, row["run_number"], worst))

plt.tight_layout()
plt.savefig("transmission_spectra_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print("panel                     run      max T in band")
for label, run_no, worst in report:
    print("  %-22s  %-7s  %s" % (label, run_no if run_no else "MISSING",
                                 "%.4f" % worst if worst else "-"))